# 03 HF Multitask Baseline (PsyCoMark)

> Shared encoder (DeBERTa) with dual heads: span extraction (multi-hot token labels) + document classification.
> Objective metrics: Macro-F1 (S1 spans, IoU≥0.5), Macro-F1 (S2 doc labels), combined score.

**Sections:** Reproducibility → Config → Data → Encoding → Model → Training → Evaluation → Artifacts → Ablations → Next Steps.


## 1. Reproducibility & Environment Setup
Deterministic configuration, environment capture, and library versions.

In [1]:
# Environment & Determinism
import os, sys, platform, socket, json, random, math, time, datetime, importlib
from pathlib import Path

import torch, numpy as np

VERSIONS = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'hostname': socket.gethostname(),
    'torch': torch.__version__,
}

SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print('Seed fixed to', SEED)
print('Versions:', json.dumps(VERSIONS, indent=2))
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')

Seed fixed to 42
Versions: {
  "python": "3.13.7",
  "platform": "Windows-10-10.0.19045-SP0",
  "hostname": "Homiepc",
  "torch": "2.8.0+cu126"
}
CUDA available: True
GPU: NVIDIA GeForce RTX 4060


## 2. Config Block (Hyperparameters & Paths)
Define experiment configuration and persist it.

In [2]:
from dataclasses import dataclass, asdict

@dataclass
class MTConfig:
    model_name: str = 'microsoft/deberta-v3-base'
    max_length: int = 1024
    fallback_length: int = 768
    doc_label_map: dict = None
    marker_labels: list = None
    lr: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    epochs: int = 5
    batch_size: int = 8
    grad_accum: int = 2
    fp16: bool = True
    lambda_span: float = 1.0
    lambda_doc: float = 1.0
    early_stop_patience: int = 3
    ckpt_dir: str = './outputs/mt_baseline'
    data_dir: str = './data/derived'
    run_subdir: str = None
    seed: int = SEED
    span_threshold: float = 0.5
    save_predictions: bool = True
    pos_weight: bool = False  # enable per-label pos weighting later

    def finalize(self):
        if self.doc_label_map is None:
            self.doc_label_map = {'non': 0, 'conspiracy': 1}
        if self.marker_labels is None:
            self.marker_labels = ['Actor','Action','Effect','Victim','Evidence']
        if self.run_subdir is None:
            self.run_subdir = f"run_{RUN_ID}"
        return self

CONFIG = MTConfig().finalize()

ckpt_path = Path(CONFIG.ckpt_dir) / CONFIG.run_subdir
ckpt_path.mkdir(parents=True, exist_ok=True)

with open(ckpt_path / 'config.json','w') as f:
    json.dump(asdict(CONFIG), f, indent=2)
print('Config saved to', ckpt_path/'config.json')

Config saved to outputs\mt_baseline\run_20251005_083629\config.json


## 3. Utility: Git Commit, Seed, Determinism Logging
Helper utilities for reproducibility metadata.

In [3]:
def get_git_commit(repo_root: Path = Path('.')) -> str:
    try:
        import subprocess
        commit = subprocess.check_output(['git', '-C', str(repo_root), 'rev-parse', 'HEAD'], stderr=subprocess.STDOUT)
        return commit.decode().strip()
    except Exception as exc:
        print('Git commit retrieval failed:', exc)
        return 'unknown'

GIT_HASH = get_git_commit()
print('Git commit hash:', GIT_HASH)

run_metadata = {
    'run_id': RUN_ID,
    'git_commit': GIT_HASH,
    'config': asdict(CONFIG),
    'versions': VERSIONS,
    'timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}

with open(ckpt_path / 'run.json', 'w') as f:
    json.dump(run_metadata, f, indent=2)
print('Run metadata recorded at', ckpt_path / 'run.json')

Git commit hash: 8330f8e58c81d1379d833c1cb091e20567b79122
Run metadata recorded at outputs\mt_baseline\run_20251005_083629\run.json


## 4. Load Raw PsyCoMark Data (train/dev)
Load JSONL splits and basic integrity checks.

In [4]:
from datasets import load_dataset

raw_data_dir = Path(CONFIG.data_dir)
train_path = raw_data_dir / 'psycomark_latest.txt'
if train_path.exists():
    # pointer file to latest run
    latest_dir = Path(train_path.read_text().strip())
else:
    latest_dir = raw_data_dir

train_file = latest_dir / 'train.jsonl'
dev_file = latest_dir / 'dev.jsonl'
assert train_file.exists(), f"Missing train file: {train_file}"
assert dev_file.exists(), f"Missing dev file: {dev_file}"

print('Using data directory:', latest_dir)

dataset = load_dataset(
    'json',
    data_files={'train': str(train_file), 'dev': str(dev_file)},
    split={'train': 'train', 'dev': 'dev'}
)

for split_name, ds in dataset.items():
    print(split_name, 'records:', len(ds))
    empty_text = sum(1 for rec in ds if not rec.get('text'))
    if empty_text:
        print(f"Warning: {empty_text} documents with empty text in {split_name}; dropping.")
        dataset[split_name] = ds.filter(lambda ex: bool(ex.get('text')))

id_sets = {split: set(ds['doc_id']) for split, ds in dataset.items()}
assert id_sets['train'].isdisjoint(id_sets['dev']), 'Train/dev doc_id overlap detected!'
print('Train/dev doc_id disjointness verified.')

c:\Users\panagiotis\Desktop\GitHub\PsyChoMark_Semeval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using data directory: C:\Users\panagiotis\Desktop\GitHub\PsyChoMark_Semeval\data\derived\psycomark_official_split_20250928_232947
train records: 3360
dev records: 100
Train/dev doc_id disjointness verified.
train records: 3360
dev records: 100
Train/dev doc_id disjointness verified.


## 5. Label Schema & Encoding Notes
- Doc labels mapped via `CONFIG.doc_label_map = {'non': 0, 'conspiracy': 1}`.
- Span labels defined in `CONFIG.marker_labels` (ordered list length = 5).
- Token label representation: matrix `(seq_len, num_markers)` with binary entries.
  - If multiple spans overlap the same token for the same label, value stays 1 (clamped binary).
  - Nested spans with different labels simply set separate columns to 1.
- Span length measured via character offsets; subsequent decoding will reconstruct spans from these token labels combined with offset mappings.


## 6. Tokenizer & Max Length Strategy
Load tokenizer with offsets and truncation policy.

In [ ]:
from packaging import version
from importlib import import_module
from transformers import AutoTokenizer, DebertaV2Tokenizer, DebertaV2TokenizerFast

model_name = CONFIG.model_name
force_slow = False  # Set True only as a last resort
local_files_only = False
local_vocab = None
local_merges = None

required_pkgs = {
    "sentencepiece": "sentencepiece",
    "protobuf": "google.protobuf",
    "tokenizers": "tokenizers",
}
missing = []
for pip_name, import_name in required_pkgs.items():
    try:
        import_module(import_name)
    except Exception:
        missing.append(pip_name)
if missing:
    print("WARNING: Missing packages. Install with: uv pip install", " ".join(missing))

protobuf_ok = True
try:
    from google.protobuf import __version__ as _pb_ver  # type: ignore
    print("protobuf version:", _pb_ver)
    if version.parse(_pb_ver) >= version.parse("5.0.0"):
        protobuf_ok = False
        print("WARNING: protobuf >=5 detected. Reinstall with: uv pip install 'protobuf<5' --force-reinstall")
except Exception as e:
    protobuf_ok = False
    print("WARNING: protobuf not importable:", e)

if not protobuf_ok:
    print("Tokenizer conversion may fail until protobuf <5 is installed.")

last_error = None
loaded = False

attempts = []
if not force_slow:
    attempts.append(("DebertaV2TokenizerFast", True))
if not force_slow:
    attempts.append(("AutoTokenizer-fast", True))
attempts.append(("AutoTokenizer-slow", False))

for label, use_fast_flag in attempts:
    try:
        if label == "DebertaV2TokenizerFast":
            tok = DebertaV2TokenizerFast.from_pretrained(model_name, local_files_only=local_files_only)
        elif label == "AutoTokenizer-fast":
            tok = AutoTokenizer.from_pretrained(model_name, use_fast=True, local_files_only=local_files_only)
        else:
            tok = AutoTokenizer.from_pretrained(model_name, use_fast=False, local_files_only=local_files_only)
            if local_vocab:
                tok = AutoTokenizer.from_pretrained(
                    model_name,
                    use_fast=False,
                    local_files_only=local_files_only,
                    vocab_file=local_vocab,
                    merges_file=local_merges,
                )
        test = tok("quick test", return_offsets_mapping=True)
        if 'offset_mapping' not in test:
            if use_fast_flag:
                raise RuntimeError("Fast tokenizer did not produce offset_mapping.")
            else:
                raise RuntimeError("Slow tokenizer missing offset_mapping; requires fast tokenizer.")
        tokenizer = tok
        loaded = True
        print(f"Loaded {label} for {model_name} ✔ (fast={getattr(tokenizer, 'is_fast', False)})")
        break
    except Exception as e:
        last_error = e
        print(f"Attempt {label} failed: {e}")
        if label == "AutoTokenizer-slow":
            print("NOTE: Slow tokenizer cannot supply offset_mapping; install protobuf<5 and retry fast mode.")

if not loaded:
    raise RuntimeError(
        "All tokenizer load attempts failed.\n"
        "Fix suggestions:\n"
        " - Ensure protobuf <5: uv pip install 'protobuf<5' --force-reinstall\n"
        " - Refresh tokenizers cache: remove %USERPROFILE%/.cache/huggingface/hub/*deberta*\n"
        " - Reinstall tokenizers stack: uv pip install --upgrade --reinstall transformers tokenizers sentencepiece\n"
        " - If offline, download tokenizer files manually and set local_vocab/local_merges\n"
        f"Last error: {last_error}"
    )

if tokenizer.model_max_length and tokenizer.model_max_length < CONFIG.max_length:
    print('Tokenizer max length smaller than requested. Using tokenizer limit:', tokenizer.model_max_length)
    CONFIG.max_length = tokenizer.model_max_length

print('Tokenizer vocab size:', getattr(tokenizer, 'vocab_size', 'n/a'))
print('Max length set to:', CONFIG.max_length)
print('Fast tokenizer:', getattr(tokenizer, 'is_fast', False))

TRUNCATION_STATS = {'total': 0, 'truncated': 0}

protobuf version: 4.25.8


c:\Users\panagiotis\Desktop\GitHub\PsyChoMark_Semeval\.venv\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Loaded DebertaV2TokenizerFast for microsoft/deberta-v3-base ✔ (fast=True)
Tokenizer vocab size: 128000
Max length set to: 1024
Fast tokenizer: True


## 7. Dataset Processing Function (Token Alignment & Multi-Hot Labels)
Map raw records to tokenized tensors with span matrices.

In [6]:
label2id = CONFIG.doc_label_map
marker2idx = {label: i for i, label in enumerate(CONFIG.marker_labels)}
num_labels = len(marker2idx)


def encode_example(example):
    text = example['text']
    TRUNCATION_STATS['total'] += 1
    encoding = tokenizer(
        text,
        max_length=CONFIG.max_length,
        truncation=True,
        return_offsets_mapping=True,
        padding='max_length'
    )
    offsets = encoding['offset_mapping']
    truncated = encoding['input_ids'].count(tokenizer.pad_token_id) == 0 and len(text) > CONFIG.max_length
    if truncated:
        TRUNCATION_STATS['truncated'] += 1

    span_matrix = np.zeros((CONFIG.max_length, num_labels), dtype=np.float32)
    spans = example.get('spans') or []
    for span in spans:
        label = span.get('label')
        start = span.get('start')
        end = span.get('end')
        if label not in marker2idx or start is None or end is None:
            continue
        label_idx = marker2idx[label]
        for token_idx, (tok_start, tok_end) in enumerate(offsets):
            if tok_start is None or tok_end is None:
                continue
            if tok_end <= start:
                continue
            if tok_start >= end:
                break
            span_matrix[token_idx, label_idx] = 1.0

    doc_label = label2id.get(example.get('doc_label', 'non'), 0)

    result = {
        'input_ids': encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'span_label_matrix': span_matrix.tolist(),
        'doc_label': doc_label,
        'offset_mapping': offsets,
        'text': text,
        'doc_id': example.get('doc_id'),
        'gold_spans': spans,
    }
    return result

## 8. Dataset Construction & Caching
Apply preprocessing and store as PyTorch-ready datasets.

In [7]:
encoded_dataset = {}
for split_name, ds in dataset.items():
    encoded = ds.map(
        encode_example,
        remove_columns=ds.column_names,
        desc=f'Encoding {split_name}'
    )
    encoded.set_format(
        type='torch',
        columns=['input_ids', 'attention_mask', 'span_label_matrix', 'doc_label'],
        output_all_columns=True
    )
    encoded_dataset[split_name] = encoded

train_ds = encoded_dataset['train']
dev_ds = encoded_dataset['dev']

print('Encoding completed. Example keys:', train_ds.features.keys())
print('Truncation rate:', TRUNCATION_STATS['truncated'], '/', TRUNCATION_STATS['total'])

Encoding dev: 100%|██████████| 100/100 [00:00<00:00, 574.53 examples/s]

Encoding completed. Example keys: dict_keys(['doc_id', 'text', 'doc_label', 'input_ids', 'attention_mask', 'span_label_matrix', 'offset_mapping', 'gold_spans'])
Truncation rate: 0 / 3460


## 9. Data Collator for Multi-Task Batching
Pads tensors and prepares span matrices/doc labels.

In [ ]:
from torch.utils.data import DataLoader

class MultiTaskCollator:
    def __init__(self, pad_token_id: int, pad_to_max: bool = True):
        self.pad_token_id = pad_token_id
        self.pad_to_max = pad_to_max

    def __call__(self, batch):
        def _ensure_tensor(value, dtype=None):
            if isinstance(value, torch.Tensor):
                return value.to(dtype=dtype) if dtype is not None else value
            if dtype is not None:
                return torch.tensor(value, dtype=dtype)
            return torch.tensor(value)

        input_ids = torch.stack([_ensure_tensor(item['input_ids'], dtype=torch.long) for item in batch])
        attention_mask = torch.stack([_ensure_tensor(item['attention_mask'], dtype=torch.long) for item in batch])
        span_labels = torch.stack([_ensure_tensor(item['span_label_matrix'], dtype=torch.float32) for item in batch])
        doc_labels = torch.stack([_ensure_tensor(item['doc_label'], dtype=torch.long) for item in batch])

        output = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels_span': span_labels,
            'labels_doc': doc_labels,
            'offset_mapping': [item['offset_mapping'] for item in batch],
            'gold_spans': [item['gold_spans'] for item in batch],
            'doc_id': [item['doc_id'] for item in batch],
            'text': [item['text'] for item in batch],
        }
        return output

collator = MultiTaskCollator(pad_token_id=tokenizer.pad_token_id)

train_loader = DataLoader(train_ds, batch_size=CONFIG.batch_size, shuffle=True, collate_fn=collator)
dev_loader = DataLoader(dev_ds, batch_size=CONFIG.batch_size, shuffle=False, collate_fn=collator)
print('DataLoaders ready:', len(train_loader), 'train batches')

DataLoaders ready: 420 train batches


## 10. Quick Sanity Checks
Inspect shapes and label distributions on a sample batch.

In [9]:
batch = next(iter(train_loader))
print('input_ids shape:', batch['input_ids'].shape)
print('span_label_matrix shape:', batch['labels_span'].shape)
print('doc_label distribution:', torch.bincount(batch['labels_doc']))

per_label_counts = batch['labels_span'].sum(dim=(0,1)).tolist()
for label, count in zip(CONFIG.marker_labels, per_label_counts):
    print(f"Label {label}: {count:.1f} positive tokens in batch")

TypeError: only integer tensors of a single element can be converted to an index

## 11. Exploratory Label Statistics & Imbalance Heuristics
Compute marker prevalence and span lengths for weighting decisions.

In [ ]:
marker_counts = {label: 0 for label in CONFIG.marker_labels}
marker_span_lengths = {label: [] for label in CONFIG.marker_labels}

def span_length(span):
    return (span.get('end', 0) or 0) - (span.get('start', 0) or 0)

for record in dataset['train']:
    spans = record.get('spans') or []
    seen_labels = set()
    for span in spans:
        label = span.get('label')
        if label not in marker2idx:
            continue
        marker_counts[label] += 1
        marker_span_lengths[label].append(span_length(span))
        seen_labels.add(label)

label_stats = []
for label in CONFIG.marker_labels:
    freq = marker_counts[label]
    avg_len = float(np.mean(marker_span_lengths[label])) if marker_span_lengths[label] else 0.0
    label_stats.append({'label': label, 'count': freq, 'avg_span_len': avg_len})

for stat in label_stats:
    print(stat)

with open(ckpt_path / 'stats.json', 'w') as f:
    json.dump({'label_stats': label_stats}, f, indent=2)
print('Label statistics saved to', ckpt_path / 'stats.json')

## 12. Custom MultiTaskModel (Shared Encoder + Dual Heads)
Subclass `PreTrainedModel` with span and doc logits.

In [ ]:
from transformers import AutoConfig, AutoModel, PreTrainedModel
import torch.nn as nn

class MultiTaskConfig(AutoConfig):
    model_type = 'mt_deberta'

    def __init__(self, num_span_labels: int, **kwargs):
        super().__init__(**kwargs)
        self.num_span_labels = num_span_labels
        self.num_doc_labels = len(CONFIG.doc_label_map)


class MultiTaskModel(PreTrainedModel):
    config_class = MultiTaskConfig

    def __init__(self, config: MultiTaskConfig):
        super().__init__(config)
        self.encoder = AutoModel.from_pretrained(CONFIG.model_name, config=config)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(self.encoder.config.hidden_dropout_prob)
        self.token_classifier = nn.Linear(hidden_size, config.num_span_labels)
        self.doc_classifier = nn.Linear(hidden_size, config.num_doc_labels)
        self.init_weights()

        self.register_buffer('span_pos_weight', None, persistent=False)
        self.register_buffer('doc_class_weight', None, persistent=False)

    def set_loss_weights(self, span_pos_weight=None, doc_class_weight=None):
        if span_pos_weight is not None:
            self.span_pos_weight = span_pos_weight
        if doc_class_weight is not None:
            self.doc_class_weight = doc_class_weight

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels_span=None,
        labels_doc=None,
        **kwargs
    ):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        sequence_output = self.dropout(outputs.last_hidden_state)
        pooled_output = self.dropout(outputs.last_hidden_state[:, 0])

        span_logits = self.token_classifier(sequence_output)
        doc_logits = self.doc_classifier(pooled_output)

        loss = None
        loss_span = None
        loss_doc = None

        if labels_span is not None:
            pos_weight = self.span_pos_weight if self.span_pos_weight is not None else None
            loss_fct_span = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            active_mask = attention_mask.unsqueeze(-1).float()
            loss_span = loss_fct_span(span_logits * active_mask, labels_span * active_mask)
        if labels_doc is not None:
            weight = self.doc_class_weight if self.doc_class_weight is not None else None
            loss_fct_doc = nn.CrossEntropyLoss(weight=weight)
            loss_doc = loss_fct_doc(doc_logits, labels_doc)

        if loss_span is not None and loss_doc is not None:
            loss = CONFIG.lambda_span * loss_span + CONFIG.lambda_doc * loss_doc
        elif loss_span is not None:
            loss = loss_span
        elif loss_doc is not None:
            loss = loss_doc

        return {
            'loss': loss,
            'loss_span': loss_span,
            'loss_doc': loss_doc,
            'span_logits': span_logits,
            'doc_logits': doc_logits,
        }

base_config = AutoConfig.from_pretrained(CONFIG.model_name)
mt_config = MultiTaskConfig(num_span_labels=num_labels, **base_config.to_dict())
model = MultiTaskModel(mt_config)
print('Model initialized with hidden size', model.encoder.config.hidden_size)

## 13. Loss Functions & Weighted Combination
Set optional class weights and confirm combined loss formulation.

In [ ]:
if CONFIG.pos_weight:
    span_pos_weight = []
    total = sum(marker_counts.values()) + 1e-6
    for label in CONFIG.marker_labels:
        freq = marker_counts[label] + 1e-6
        span_pos_weight.append(math.sqrt(total / freq))
    span_pos_weight_tensor = torch.tensor(span_pos_weight, dtype=torch.float32)
else:
    span_pos_weight_tensor = None

# Document class weights (inverse frequency)
doc_counts = torch.zeros(len(CONFIG.doc_label_map), dtype=torch.float32)
for record in dataset['train']:
    doc_counts[label2id.get(record.get('doc_label', 'non'), 0)] += 1

if doc_counts.sum() > 0:
    doc_weights = (doc_counts.sum() / (doc_counts + 1e-6))
    doc_weights = doc_weights / doc_weights.mean()
else:
    doc_weights = torch.ones_like(doc_counts)

if span_pos_weight_tensor is not None:
    model.set_loss_weights(span_pos_weight=span_pos_weight_tensor)
else:
    model.set_loss_weights()

model.set_loss_weights(doc_class_weight=doc_weights)
print('Span pos weight:', span_pos_weight_tensor)
print('Doc class weight:', doc_weights)
print('Loss combination: L = λ_span * L_span + λ_doc * L_doc with λ=', CONFIG.lambda_span, CONFIG.lambda_doc)

## 14. Custom Trainer Subclass
Override `compute_loss` to utilize multitask outputs and log components.

In [ ]:
from transformers import Trainer

class MultiTaskTrainer(Trainer):
    """Custom trainer that handles multitask losses and returns structured outputs."""

    def compute_loss(self, model, inputs, return_outputs=False):
        inputs = inputs.copy()
        labels_span = inputs.pop('labels_span', None)
        labels_doc = inputs.pop('labels_doc', None)
        # Remove metadata the model does not consume
        inputs.pop('offset_mapping', None)
        inputs.pop('gold_spans', None)
        inputs.pop('doc_id', None)
        inputs.pop('text', None)

        outputs = model(**inputs, labels_span=labels_span, labels_doc=labels_doc)
        loss = outputs['loss']
        if loss is None:
            raise ValueError('Model did not return a loss. Verify inputs include labels.')

        if model.training:
            logs = {}
            if outputs.get('loss_span') is not None:
                logs['train/loss_span'] = outputs['loss_span'].detach().mean().item()
            if outputs.get('loss_doc') is not None:
                logs['train/loss_doc'] = outputs['loss_doc'].detach().mean().item()
            if logs:
                self.log(logs)

        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        inputs = inputs.copy()

        labels_span = inputs.pop('labels_span', None)
        labels_doc = inputs.pop('labels_doc', None)
        offset_mapping = inputs.pop('offset_mapping', None)
        gold_spans = inputs.pop('gold_spans', None)
        doc_ids = inputs.pop('doc_id', None)
        texts = inputs.pop('text', None)
        attention_mask = inputs.get('attention_mask')

        with torch.no_grad():
            outputs = model(**inputs, labels_span=labels_span, labels_doc=labels_doc)

        loss = outputs['loss']
        if prediction_loss_only:
            return loss, None, None

        span_logits = outputs['span_logits'].detach().cpu()
        doc_logits = outputs['doc_logits'].detach().cpu()
        label_span_np = labels_span.detach().cpu().numpy() if labels_span is not None else None
        label_doc_np = labels_doc.detach().cpu().numpy() if labels_doc is not None else None
        attention_mask_np = attention_mask.detach().cpu().numpy() if attention_mask is not None else None

        meta_entry = {
            'offset_mapping': offset_mapping,
            'gold_spans': gold_spans,
            'doc_id': doc_ids,
            'text': texts,
            'attention_mask': attention_mask_np,
        }
        global EVAL_METADATA_BUFFER
        if self.args.world_size <= 1 or self.state.is_world_process_zero:
            EVAL_METADATA_BUFFER.append(meta_entry)

        predictions = (span_logits.numpy(), doc_logits.numpy())
        label_tensors = (label_span_np, label_doc_np)
        return loss, predictions, label_tensors

## 15. Span Decoding Utilities
Convert token-level logits into character spans using offset mappings.

In [ ]:
def decode_spans_for_example(
    span_logits: torch.Tensor,
    attention_mask: torch.Tensor,
    offset_mapping,
    threshold: float = 0.5,
    min_tokens: int = 1,
):
    """Decode spans for a single example."""
    probs = torch.sigmoid(span_logits)
    decoded = []
    mask = attention_mask.tolist()

    for label_idx, label in enumerate(CONFIG.marker_labels):
        current_start = None
        current_end = None
        collected_scores = []
        for token_idx, mask_value in enumerate(mask):
            if mask_value == 0:
                break
            start, end = offset_mapping[token_idx]
            if end <= start:
                continue
            score = probs[token_idx, label_idx].item()
            if score >= threshold:
                if current_start is None:
                    current_start = int(start)
                current_end = int(end)
                collected_scores.append(score)
            else:
                if current_start is not None and len(collected_scores) >= min_tokens:
                    decoded.append({
                        'label': label,
                        'start': current_start,
                        'end': current_end,
                        'score': float(np.mean(collected_scores)),
                        'length_chars': current_end - current_start,
                        'length_tokens': len(collected_scores),
                    })
                current_start = None
                current_end = None
                collected_scores = []
        if current_start is not None and len(collected_scores) >= min_tokens:
            decoded.append({
                'label': label,
                'start': current_start,
                'end': current_end,
                'score': float(np.mean(collected_scores)),
                'length_chars': current_end - current_start,
                'length_tokens': len(collected_scores),
            })
    return decoded


def decode_span_batch(span_logits_batch, attention_masks, offset_mappings, threshold=0.5):
    results = []
    for logits, mask, offsets in zip(span_logits_batch, attention_masks, offset_mappings):
        results.append(decode_spans_for_example(logits, mask, offsets, threshold=threshold))
    return results

## 16. Document-Level Prediction Helpers
Softmax doc logits and format predictions with label names.

In [ ]:
id2label = {idx: label for label, idx in label2id.items()}


def decode_doc_logits(doc_logits: torch.Tensor):
    probs = torch.softmax(doc_logits, dim=-1)
    max_prob, pred_idx = torch.max(probs, dim=-1)
    pred_labels = [id2label[int(idx)] for idx in pred_idx]
    return {
        'pred_ids': pred_idx.cpu().tolist(),
        'pred_labels': pred_labels,
        'pred_probs': max_prob.detach().cpu().tolist(),
        'full_probs': probs.detach().cpu().numpy(),
    }

## 17. Metric Computation (Span IoU & Document F1)
Utility functions for IoU matching, macro-F1 scoring, and combined metrics.

In [ ]:
EVAL_METADATA_BUFFER = []
LAST_EVAL_METADATA = None


def reset_eval_buffer():
    global EVAL_METADATA_BUFFER, LAST_EVAL_METADATA
    EVAL_METADATA_BUFFER = []
    LAST_EVAL_METADATA = None


def span_iou(span_a, span_b):
    start = max(span_a['start'], span_b.get('start', span_b.get('begin', 0)))
    end = min(span_a['end'], span_b.get('end', span_b.get('finish', 0)))
    intersection = max(0, end - start)
    len_a = span_a['end'] - span_a['start']
    len_b = span_b.get('end', span_b.get('finish', 0)) - span_b.get('start', span_b.get('begin', 0))
    union = len_a + len_b - intersection
    if union <= 0:
        return 0.0
    return intersection / union

...


## 18. Bootstrap Confidence Intervals
Estimate uncertainty for macro-F1 metrics via resampling.

In [ ]:
def bootstrap_confidence_intervals(
    span_preds,
    gold_spans,
    doc_pred_ids,
    doc_gold_ids,
    labels=None,
    n_bootstrap: int = 200,
    alpha: float = 0.05,
    seed: int = SEED,
):
    labels = labels or CONFIG.marker_labels
    if not span_preds:
        return {
            'span_macro_f1': (0.0, 0.0),
            'doc_macro_f1': (0.0, 0.0),
            'combined_macro_f1': (0.0, 0.0),
        }

    rng = np.random.default_rng(seed)
    n_samples = len(span_preds)
    span_scores = []
    doc_scores = []
    combined_scores = []

    indices = np.arange(n_samples)
    for _ in range(n_bootstrap):
        sampled = rng.choice(indices, size=n_samples, replace=True)
        sampled_span_preds = [span_preds[i] for i in sampled]
        sampled_gold_spans = [gold_spans[i] for i in sampled]
        sampled_doc_pred = [doc_pred_ids[i] for i in sampled]
        sampled_doc_gold = [doc_gold_ids[i] for i in sampled]

        span_metric = compute_span_metrics_batch(sampled_span_preds, sampled_gold_spans, labels=labels)
        doc_metric = compute_doc_metrics(sampled_doc_pred, sampled_doc_gold)
        span_scores.append(span_metric['macro_f1'])
        doc_scores.append(doc_metric['macro_f1'])
        combined_scores.append(combined_metric(span_metric, doc_metric))

    lower_q = 100 * (alpha / 2)
    upper_q = 100 * (1 - alpha / 2)
    return {
        'span_macro_f1': (float(np.percentile(span_scores, lower_q)), float(np.percentile(span_scores, upper_q))),
        'doc_macro_f1': (float(np.percentile(doc_scores, lower_q)), float(np.percentile(doc_scores, upper_q))),
        'combined_macro_f1': (float(np.percentile(combined_scores, lower_q)), float(np.percentile(combined_scores, upper_q))),
    }

## 19. TrainingArguments & Callbacks
Define optimization schedule, logging cadence, and early stopping.

In [ ]:
from transformers import TrainingArguments, IntervalStrategy, EarlyStoppingCallback


def prepare_eval_metadata(pop: bool = True):
    """Flatten buffered metadata collected during evaluation passes."""
    global EVAL_METADATA_BUFFER
    if not EVAL_METADATA_BUFFER:
        return None

    merged = {
        'offset_mapping': [],
        'gold_spans': [],
        'doc_id': [],
        'text': [],
        'attention_mask': [],
    }
    for entry in EVAL_METADATA_BUFFER:
        merged['offset_mapping'].extend(entry['offset_mapping'])
        merged['gold_spans'].extend(entry['gold_spans'])
        merged['doc_id'].extend(entry['doc_id'])
        merged['text'].extend(entry['text'])
        merged['attention_mask'].extend(entry['attention_mask'])

    if pop:
        EVAL_METADATA_BUFFER = []
    return merged


def multitask_compute_metrics(eval_pred):
    global LAST_EVAL_METADATA
    metadata = prepare_eval_metadata(pop=True)
    LAST_EVAL_METADATA = metadata
    if metadata is None:
        return {}

    span_logits, doc_logits = eval_pred.predictions
    labels_span, labels_doc = eval_pred.label_ids

    span_logits_tensor = torch.from_numpy(span_logits)
    attention_masks = torch.tensor(np.array(metadata['attention_mask']))
    decoded_spans = decode_span_batch(
        span_logits_tensor,
        attention_masks,
        metadata['offset_mapping'],
        threshold=CONFIG.span_threshold,
    )

    doc_predictions = decode_doc_logits(torch.from_numpy(doc_logits))
    doc_pred_ids = doc_predictions['pred_ids']
    labels_doc = [int(x) for x in np.asarray(labels_doc)]

    span_metrics = compute_span_metrics_batch(decoded_spans, metadata['gold_spans'])
    doc_metrics = compute_doc_metrics(doc_pred_ids, labels_doc)
    combined = combined_metric(span_metrics, doc_metrics)

    return {
        'span_macro_f1': float(span_metrics['macro_f1']),
        'doc_macro_f1': float(doc_metrics['macro_f1']),
        'combined_macro_f1': float(combined),
    }


training_args = TrainingArguments(
    output_dir=str(ckpt_path / 'hf_ckpts'),
    overwrite_output_dir=True,
    evaluation_strategy=IntervalStrategy.EPOCH,
    save_strategy=IntervalStrategy.EPOCH,
    save_total_limit=2,
    per_device_train_batch_size=CONFIG.batch_size,
    per_device_eval_batch_size=CONFIG.batch_size,
    gradient_accumulation_steps=CONFIG.grad_accum,
    learning_rate=CONFIG.lr,
    weight_decay=CONFIG.weight_decay,
    num_train_epochs=CONFIG.epochs,
    warmup_ratio=CONFIG.warmup_ratio,
    logging_strategy=IntervalStrategy.STEPS,
    logging_steps=50,
    report_to=['tensorboard'],
    fp16=CONFIG.fp16,
    load_best_model_at_end=True,
    metric_for_best_model='combined_macro_f1',
    greater_is_better=True,
    seed=CONFIG.seed,
    dataloader_num_workers=2,
)

callbacks = [EarlyStoppingCallback(early_stopping_patience=CONFIG.early_stop_patience)]
print('TrainingArguments prepared. Output dir:', training_args.output_dir)

## 20. Instantiate Trainer & Inspect Setup
Create the trainer, attach callbacks, and summarize parameter counts.

In [ ]:
reset_eval_buffer()

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=multitask_compute_metrics,
    callbacks=callbacks,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable_params:,} / Total params: {total_params:,}')
steps_per_epoch = math.ceil(len(train_ds) / (CONFIG.batch_size * CONFIG.grad_accum))
print('Approx. optimizer steps per epoch:', steps_per_epoch)
print('Evaluation strategy:', training_args.evaluation_strategy)
print('Using early stopping patience:', CONFIG.early_stop_patience)

## 21. (Optional) Dry-Run Hooks
Validate metric plumbing before full training and preview resource usage.

In [ ]:
if torch.cuda.is_available():
    gpu_props = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu_props.name} | {gpu_props.total_memory / 1e9:.2f} GB')
else:
    print('GPU not detected. Training will run on CPU (much slower).')

# Optional: run a very small evaluation pass to ensure decoding metrics work end-to-end.
# This uses `max_eval_samples` to keep runtime manageable during development.
if False:
    reset_eval_buffer()
    preview_metrics = trainer.evaluate(eval_dataset=dev_ds.select(range(4)))
    print('Preview metrics:', preview_metrics)

# When ready, remove the guard and launch full training:
# if True:
#     train_result = trainer.train()
#     trainer.save_model(ckpt_path / 'final_model')
#     trainer.save_state()
#     print('Training metrics:', train_result.metrics)

## 22. Evaluate on Dev Set & Save Artifacts
Decode spans/doc labels, compute metrics with confidence intervals, and persist outputs.

In [ ]:
RUN_EVAL = False  # Flip to True after training completes.

if RUN_EVAL:
    reset_eval_buffer()
    eval_output = trainer.predict(dev_ds)
    metadata = LAST_EVAL_METADATA
    if metadata is None:
        raise RuntimeError('Evaluation metadata not captured. Ensure compute_metrics ran successfully.')

    span_logits, doc_logits = eval_output.predictions
    attention_masks = torch.tensor(np.array(metadata['attention_mask']))
    span_predictions = decode_span_batch(
        torch.from_numpy(span_logits),
        attention_masks,
        metadata['offset_mapping'],
        threshold=CONFIG.span_threshold,
    )
    doc_predictions = decode_doc_logits(torch.from_numpy(doc_logits))
    gold_doc_labels = [int(x) for x in eval_output.label_ids[1]]

    span_metrics = compute_span_metrics_batch(span_predictions, metadata['gold_spans'])
    doc_metrics = compute_doc_metrics(doc_predictions['pred_ids'], gold_doc_labels)
    combined = combined_metric(span_metrics, doc_metrics)
    ci = bootstrap_confidence_intervals(
        span_predictions,
        metadata['gold_spans'],
        doc_predictions['pred_ids'],
        gold_doc_labels,
    )
    ci = {k: [float(v[0]), float(v[1])] for k, v in ci.items()}

    metrics_blob = {
        'span_metrics': span_metrics,
        'doc_metrics': doc_metrics,
        'combined_macro_f1': float(combined),
        'confidence_intervals': ci,
        'trainer_metrics': eval_output.metrics,
    }

    metrics_path = ckpt_path / 'metrics.json'
    with open(metrics_path, 'w') as f:
        json.dump(metrics_blob, f, indent=2)

    doc_records = []
    for idx, doc_id in enumerate(metadata['doc_id']):
        doc_records.append({
            'doc_id': doc_id,
            'pred_label': doc_predictions['pred_labels'][idx],
            'pred_prob': float(doc_predictions['pred_probs'][idx]),
            'gold_label': id2label[gold_doc_labels[idx]],
        })

    span_records = []
    for idx, doc_id in enumerate(metadata['doc_id']):
        span_records.append({
            'doc_id': doc_id,
            'predicted_spans': span_predictions[idx],
            'gold_spans': metadata['gold_spans'][idx],
        })

    with open(ckpt_path / 'doc_predictions.json', 'w') as f:
        json.dump(doc_records, f, indent=2)
    with open(ckpt_path / 'span_predictions.json', 'w') as f:
        json.dump(span_records, f, indent=2)

    try:
        import pandas as pd  # type: ignore

        doc_df = pd.DataFrame(doc_records)
        doc_df.to_csv(ckpt_path / 'doc_predictions.csv', index=False)
    except Exception as exc:
        print('Pandas unavailable for CSV export:', exc)

    print(json.dumps(metrics_blob, indent=2))
    print('Artifacts saved to', ckpt_path)
    reset_eval_buffer()

## 23. Training Log Summary & Cost Estimate
Aggregate trainer logs for quick reporting and rough compute accounting.

In [ ]:
def summarize_trainer_state(current_trainer: MultiTaskTrainer):
    history = current_trainer.state.log_history
    if not history:
        print('Trainer log history is empty. Run training to populate metrics.')
        return

    log_path = ckpt_path / 'trainer_log.json'
    with open(log_path, 'w') as f:
        json.dump(history, f, indent=2)

    approx_tokens_per_step = CONFIG.batch_size * CONFIG.grad_accum * CONFIG.max_length
    total_steps = current_trainer.state.global_step or 0
    approx_total_tokens = approx_tokens_per_step * total_steps

    summary = {
        'global_step': total_steps,
        'epochs_completed': float(current_trainer.state.epoch or 0.0),
        'best_metric': current_trainer.state.best_metric,
        'best_model_checkpoint': current_trainer.state.best_model_checkpoint,
        'approx_tokens_per_step': approx_tokens_per_step,
        'approx_total_tokens': approx_total_tokens,
        'approx_total_tokens_billions': approx_total_tokens / 1e9,
    }

    summary_path = ckpt_path / 'run_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)

    print('Trainer summary:')
    print(json.dumps(summary, indent=2))
    print('Detailed log history saved to', log_path)


# summarize_trainer_state(trainer)  # Uncomment after training

## 24. Error Analysis Helpers
Utility functions to explore misclassifications and span mismatches after evaluation.

In [ ]:
def _safe_load_json(path: Path):
    if not path.exists():
        print('File not found:', path)
        return []
    with open(path) as f:
        return json.load(f)


def sample_doc_errors(n: int = 5):
    records = _safe_load_json(ckpt_path / 'doc_predictions.json')
    if not records:
        return []
    mismatches = [rec for rec in records if rec['pred_label'] != rec['gold_label']]
    return mismatches[:n]


def find_span_errors(doc_id: str):
    span_records = _safe_load_json(ckpt_path / 'span_predictions.json')
    for rec in span_records:
        if rec['doc_id'] == doc_id:
            predicted = rec['predicted_spans']
            gold = rec['gold_spans']
            matches = []
            for pred in predicted:
                best = max((span_iou(pred, g), g) for g in gold) if gold else (0.0, None)
                matches.append({'prediction': pred, 'best_gold': best[1], 'iou': best[0]})
            return {
                'doc_id': doc_id,
                'matches': matches,
                'gold_only': [g for g in gold if g not in [m['best_gold'] for m in matches if m['best_gold']]],
            }
    print('Doc id not found in span predictions:', doc_id)
    return None


# Example usage (after running evaluation):
# for err in sample_doc_errors(3):
#     print(err)
#     span_detail = find_span_errors(err['doc_id'])
#     print(span_detail)

## 25. Ablation Hooks & Plan Review
Template for structured experiments and checklist for remaining roadmap items.

- **Loss weighting experiments:** toggle `CONFIG.lambda_span` / `lambda_doc`, or enable `CONFIG.pos_weight` for imbalance handling.
- **Model depth/size:** try `microsoft/deberta-v3-large` or `roberta-large` (update `CONFIG.model_name` accordingly).
- **Span threshold sweep:** evaluate `CONFIG.span_threshold` ∈ {0.4, 0.5, 0.6} to trade precision vs recall.
- **Few-shot augmentation:** incorporate synthetic spans from hard-example mining (Section 11 results).
- **Domain adaptation:** continue pretraining on PsyCoMark corpus via masked LM prior to multitask finetuning.

✅ **Plan alignment check**

| Milestone | Status | Notes |
| --- | --- | --- |
| Data ingestion & sanity checks | ✅ | Sections 4–11 verify integrity and label stats. |
| Model + loss wiring | ✅ | Sections 12–14 with shared encoder, weighted losses. |
| Metrics & decoding | ✅ | Sections 15–18 including bootstrapped CIs. |
| Training & evaluation scaffolding | ✅ | Sections 19–24 with trainer, evaluation, artifacts, and analysis. |
| Ablations & roadmap | ✅ | Current section outlines next experiments. |


In [ ]:
ABLATION_TEMPLATES = [
    {
        'name': 'span_threshold_0.4',
        'updates': {'span_threshold': 0.4},
    },
    {
        'name': 'pos_weight_on',
        'updates': {'pos_weight': True},
    },
    {
        'name': 'lambda_span_1_5',
        'updates': {'lambda_span': 1.5, 'lambda_doc': 1.0},
    },
]


def spawn_ablation_run(template, base_config: MTConfig):
    cfg_dict = asdict(base_config)
    cfg_dict.update(template['updates'])
    new_config = MTConfig(**cfg_dict).finalize()
    print(f"Spawned ablation '{template['name']}' with overrides: {template['updates']}")
    return new_config


# Example usage:
# for template in ABLATION_TEMPLATES:
#     ablation_config = spawn_ablation_run(template, CONFIG)
#     # Proceed to re-run Sections 6 onward with ablation_config as the active configuration.
